# Feature Engineering

## Objective

The objective of this notebook is to transform the cleaned household energy consumption dataset into a feature-rich dataset suitable for anomaly detection.

Feature engineering captures temporal patterns, historical behavior, statistical characteristics, and consumption dynamics that cannot be learned effectively from raw measurements alone.

The engineered features generated in this notebook will be used as inputs for the Isolation Forest model.

In [ ]:
import pandas as pd 
import numpy as np 
pd.set_option('display.max_columns',None)

## Loading the Preprocessed Dataset

In [ ]:
data_path="C:/EcoWatt-AI/data/processed/cleaned_energy_data.csv"
df =pd.read_csv(data_path)
df.shape

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df["datetime"]= pd.to_datetime(df["datetime"])
df["datetime"].dtype

In [ ]:
print("segment_id exists:","segment_id" in df.columns)
print("Number of Segments:", df["segment_id"].nunique())
print("Missing values:",df.isnull().sum().sum())
print("Duplicate rows:",df.duplicated().sum())
print("Datetime sorted:",df["datetime"].is_monotonic_increasing)
df.isnull().sum()
df.head()

In [ ]:
if "time_gap" in df.columns:
    df = df.drop(columns =["time_gap"])

print("Dataset after removing helper column:",df.shape)
print("Total Missing values:\n",df.isnull().sum())

## Observation

The cleaned dataset obtained from the preprocessing stage serves as the input for feature engineering.

At this stage, the dataset contains reliable observations with consistent timestamps and minimal missing values, making it suitable for creating advanced machine learning features.

## Temporal Feature Engineering

In [ ]:
df["hour"] = df["datetime"].dt.hour
df["day_of_week"] = df["datetime"].dt.dayofweek
df["month"] = df["datetime"].dt.month
df["day_of_year"] = df["datetime"].dt.dayofyear
df["is_weekend"] = (df["day_of_week"] >= 5).astype(int)

In [ ]:
df[["datetime","hour","day_of_week","month","day_of_year","is_weekend"]].head(10)

## Observation

Energy consumption is strongly influenced by time.

Temporal features such as hour of the day, day of the week, month, and day of the year enable the model to recognize regular consumption cycles and seasonal variations.

## Cyclical Encoding of Temporal Features

In [ ]:
df["hour_sin"] = np.sin(2*np.pi*df["hour"]/24)
df["hour_cos"] = np.cos(2*np.pi*df["hour"]/24)
df["day_sin"] = np.sin(2*np.pi*df["day_of_week"]/7)
df["day_cos"] = np.cos(2*np.pi*df["day_of_week"]/7)
df["year_sin"] = np.sin(2*np.pi*df["day_of_year"]/365.25)
df["year_cos"] = np.cos(2*np.pi*df["day_of_year"]/365.25)


In [ ]:
df[["datetime","hour","hour_sin","hour_cos","day_of_week","day_sin","day_cos","day_of_year","year_sin","year_cos"]].head(10)

In [ ]:
cyclical_columns =["hour_sin","hour_cos","day_sin","day_cos","year_sin","year_cos"]
df[cyclical_columns].describe()

In [ ]:
df[df["hour"].isin([0,1,22,23])][["hour","hour_sin","hour_cos"]].drop_duplicates().sort_values("hour")

## Why Cyclical Encoding?

Time-based variables are naturally cyclical.

For example:

- Hour 23 and Hour 0 are consecutive hours.
- Sunday and Monday are adjacent days.

Simple numerical encoding fails to capture this relationship.

Sine and cosine transformations preserve the circular nature of time and improve the representation of periodic behaviour.

## Primary Energy Signal Selection

In [ ]:
primary_signal = "Global_active_power"
print("Primary energy signal:",primary_signal)
df[primary_signal].describe()

## Observation

Global Active Power was selected as the primary signal because it directly represents household electricity consumption.

Most historical, rolling, and behavioural features are derived from this variable since it contains the strongest information regarding consumption patterns.

## Historical Lag Features

In [ ]:
lag_periods = [1, 5, 15, 60]

for lag in lag_periods:
    df[f"active_power_lag_{lag}"] = (
        df.groupby("segment_id")[primary_signal]
        .shift(lag)
    )

In [ ]:
df[["datetime","segment_id","Global_active_power","active_power_lag_1","active_power_lag_5","active_power_lag_15","active_power_lag_60"]].head(70)

In [ ]:
lag_columns = [f"active_power_lag_{lag}" for lag in lag_periods]
df[lag_columns].isnull().sum()

In [ ]:
segment_start_rows =(df.groupby("segment_id").head(1))
segment_start_rows[["datetime","segment_id","Global_active_power","active_power_lag_1","active_power_lag_5","active_power_lag_15","active_power_lag_60"]]

## Observation

Lag features provide historical context by allowing each observation to reference previous energy consumption values.

These features help the anomaly detection model identify sudden changes, unusual spikes, and deviations from historical behaviour.

## Rolling Statistical Features

In [ ]:
historical_power = (df.groupby("segment_id")[primary_signal].shift(1))

In [ ]:
rolling_windows =[15,60]
for window in rolling_windows :
    df[f"active_power_rolling_mean_{window}"] = (historical_power.groupby(df["segment_id"]).rolling(window=window,min_periods=window).mean().reset_index(level=0,drop=True))
    df[f"active_power_rolling_std_{window}"] = (historical_power.groupby(df["segment_id"]).rolling(window=window,min_periods=window).std().reset_index(level=0,drop=True))
    df[f"active_power_rolling_min_{window}"] = (historical_power.groupby(df["segment_id"]).rolling(window=window,min_periods=window).min().reset_index(level=0,drop=True))
    df[f"active_power_rolling_max_{window}"] = (historical_power.groupby(df["segment_id"]).rolling(window=window,min_periods=window).max().reset_index(level=0,drop=True))
    

In [ ]:
rolling_columns = [
    column
    for column in df.columns
    if "rolling_" in column
]

print("Rolling feature columns:")

for column in rolling_columns:
    print(column)

In [ ]:
df[
    [
        "datetime",
        "segment_id",
        "Global_active_power",
        "active_power_rolling_mean_15",
        "active_power_rolling_std_15",
        "active_power_rolling_min_15",
        "active_power_rolling_max_15",
        "active_power_rolling_mean_60",
        "active_power_rolling_std_60"
    ]
].head(70)

In [ ]:
df[rolling_columns].isnull().sum()

In [ ]:
test_index = 100
current_segment = df.loc[
    test_index,
    "segment_id"
]

manual_history = df.loc[
    (test_index - 15):(test_index - 1),
    primary_signal
]

print(
    "Current power:",
    df.loc[test_index, primary_signal]
)

print(
    "Manual previous-15 mean:",
    manual_history.mean()
)

print(
    "Engineered rolling mean:",
    df.loc[
        test_index,
        "active_power_rolling_mean_15"
    ]
)

## Observation

Rolling statistics summarize recent energy consumption behaviour over fixed time windows.

The rolling mean represents local trends, while rolling standard deviation measures variability.

Together they provide contextual information that improves anomaly detection.

## Consumption Change and Historical Deviation Features

In [ ]:
df["active_power_change_1"] = (df[primary_signal]-df["active_power_lag_1"])

In [ ]:
epsilon = 1e-6

df["active_power_change_rate"] = (
    df["active_power_change_1"]
    / (
        df["active_power_lag_1"].abs()
        + epsilon
    )
)

In [ ]:
df["deviation_from_15min_mean"] = (df[primary_signal] - df["active_power_rolling_mean_15"])
df["deviation_from_60min_mean"] = (df[primary_signal] - df["active_power_rolling_mean_60"])

In [ ]:
df["rolling_zscore_15"] = (df["deviation_from_15min_mean"]/(df["active_power_rolling_std_15"]+ epsilon))
df["rolling_zscore_60"] = (df["deviation_from_60min_mean"]/(df["active_power_rolling_std_60"]+ epsilon))

In [ ]:
change_deviation_columns = ["active_power_change_1","active_power_change_rate","deviation_from_15min_mean","deviation_from_60min_mean","rolling_zscore_15","rolling_zscore_60"]

df[["datetime","Global_active_power"]+ change_deviation_columns].head(70)

In [ ]:
for column in change_deviation_columns:

    inf_count = np.isinf(df[column]).sum()

    print(column,"infinite values:",inf_count)

In [ ]:
df[change_deviation_columns].describe()

In [ ]:
df[["datetime","segment_id","Global_active_power","active_power_rolling_mean_15","active_power_rolling_std_15","deviation_from_15min_mean","rolling_zscore_15"]].sort_values("rolling_zscore_15",ascending=False).head(10)

In [ ]:
df[
    [
        "datetime",
        "segment_id",
        "Global_active_power",
        "active_power_lag_1",
        "active_power_change_1",
        "active_power_change_rate"
    ]
].sort_values(
    "active_power_change_rate",
    ascending=False
).head(10)

## Observation

Behavioural features quantify how quickly energy consumption changes over time.

Large deviations from historical averages often indicate unusual operating conditions, making these features highly informative for anomaly detection.

## Numerical Stabilization of Historical Z-Scores

In [ ]:
unique_power_values = np.sort(df[primary_signal].dropna().unique())

power_increments = np.diff(unique_power_values)

positive_increments = power_increments[power_increments > 1e-10]

print("Minimum positive power increment:",positive_increments.min())

print("Median positive power increment:",np.median(positive_increments))

print("1st percentile positive increment:",np.percentile(positive_increments,1))

In [ ]:
power_resolution = np.percentile(positive_increments,1)

std_floor = power_resolution

print("Estimated power resolution:",power_resolution)

print("Standard deviation floor:",std_floor)

In [ ]:
print("15-minute std below floor:",(df["active_power_rolling_std_15"]< std_floor).sum())

print("60-minute std below floor:",(df["active_power_rolling_std_60"]< std_floor).sum())

In [ ]:
stable_std_15 = (df["active_power_rolling_std_15"].clip(lower=std_floor))
stable_std_60 = (df["active_power_rolling_std_60"].clip(lower=std_floor))

In [ ]:
df["rolling_zscore_15"] = (df["deviation_from_15min_mean"]/stable_std_15)
df["rolling_zscore_60"] = (df["deviation_from_60min_mean"]/stable_std_60)

In [ ]:
df[["rolling_zscore_15","rolling_zscore_60"]].describe()

In [ ]:
df[["datetime","segment_id","Global_active_power","active_power_rolling_mean_15","active_power_rolling_std_15","deviation_from_15min_mean","rolling_zscore_15"]].sort_values("rolling_zscore_15",ascending=False).head(10)

## Observation

Very small standard deviations can produce excessively large z-score values.

Applying numerical stabilization prevents division by near-zero values and improves the robustness of the engineered features without altering their underlying behaviour.

## Signed Logarithmic Transformation

In [ ]:
def signed_log_transform(series):
    return(np.sign(series)*np.log1p(np.abs(series)))

In [ ]:
df["active_power_change_rate_log"] = (signed_log_transform(df["active_power_change_rate"]))
df["rolling_zscore_15_log"] = (signed_log_transform(df["rolling_zscore_15"]))
df["rolling_zscore_60_log"] = (signed_log_transform(df["rolling_zscore_60"]))

In [ ]:
comparison_columns = ["active_power_change_rate","active_power_change_rate_log","rolling_zscore_15","rolling_zscore_15_log","rolling_zscore_60","rolling_zscore_60_log"]

df[comparison_columns].describe()

In [ ]:
df.isnull().sum().sort_values(ascending=False).head(20)

In [ ]:
model_df = df.dropna().copy()
print("Shape before:",df.shape)
print("Shape after:",model_df.shape)

In [ ]:
print("missing values:",model_df.isnull().sum().sum())
print("Duplicate rows::",model_df.duplicated().sum())


In [ ]:
print("="*60)

print("Final Feature Engineering Summary")

print("="*60)

print("Rows:", model_df.shape[0])
print("Columns:", model_df.shape[1])

print("Date Range:")
print(model_df["datetime"].min())
print(model_df["datetime"].max())

print("Continuous Segments:",
      model_df["segment_id"].nunique())

print("="*60)

## Observation

Several engineered features exhibit highly skewed distributions.

Applying a signed logarithmic transformation compresses extreme values while preserving the direction of change.

This improves numerical stability and reduces the influence of outliers.

## Final Remarks

At this stage, the dataset contains:

- Temporal features
- Cyclical features
- Historical lag features
- Rolling statistical features
- Behavioural features
- Normalized historical deviation features
- Log-transformed features

These engineered variables provide meaningful representations of household energy consumption patterns and significantly enhance the capability of the Isolation Forest algorithm to distinguish normal behaviour from anomalies.

In [ ]:
model_df.to_csv("C:/EcoWatt-AI/data/processed/energy_features.csv",index=False)

## Next Step

The engineered dataset generated in this notebook will be used in the next stage of the project, where the features are standardized and an Isolation Forest model is trained to detect abnormal household energy consumption patterns. The trained model will later be interpreted using SHAP and deployed through the EcoWatt AI Streamlit dashboard.

# Conclusion

Feature engineering successfully transformed raw energy measurements into informative machine learning features.

The generated features capture temporal patterns, historical dependencies, local statistical behaviour, consumption dynamics, and numerical stability.

These engineered variables form the foundation of the anomaly detection model and improve its ability to identify abnormal energy consumption patterns while maintaining robustness and interpretability.

The resulting dataset is now ready for feature scaling, Isolation Forest training, SHAP explainability, and dashboard visualization.